In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def load_and_prepare_data(path):
    """Loads Titanic dataset and adds feature engineering"""
    data = pd.read_csv(path)
    data['FamilySize'] = data['Parch'] + data['SibSp'] + 1
    data['HasCabin'] = data['Cabin'].notnull().astype(int)
    data['FarePerPerson'] = data['Fare'] / data['FamilySize']
    data['Title'] = data['Name'].str.extract(' ([A-Za-z]+)\.', expand = False)
    return data

def build_pipeline(numerical_features, categorical_features, int_features):
    """Builds preprocessing pipeline for numeric, integer and categorical data."""
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy = 'median')),
        ('scaler', StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy = 'most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False))
    ])

    int_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ])

    full_pipeline = ColumnTransformer([
        ('num', num_pipeline, numerical_features),
        ('int', int_pipeline, int_features),
        ('cat', cat_pipeline, categorical_features)
    ])
    return full_pipeline

def sigmoid(z):
    """Computes the sigmoid activation function used in logistic regression."""
    return 1 / (1 + np.exp(-z))

def generate_predictions(X, weights):
    """Returns predicted probabilities for binary classification."""
    z = np.dot(X, weights)
    return sigmoid(z)

def train_logistic_regression(X, Y, iterations, lr):
    """Trains a logistic regression model using gradient descent."""
    weights = np.zeros(X.shape[1])
    
    #main loop of an algorithm
    for i in range(iterations):
        prediction = generate_predictions(X, weights)
        error = prediction - Y
        gradient = np.dot(X.T, error) / len(Y)
        weights -= lr * gradient
    return prediction, weights

def find_best_threshold(data):
    """Finds threshold that maximizes accuracy."""
    best_accuracy = 0
    best_threshold = 0
    for threshold in np.arange(0.1, 0.9, 0.01):
        predictions = (data['PredictedChance'] >= threshold).astype('int')
        accuracy = (predictions == data['Survived']).mean()
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_threshold = threshold
    return best_threshold

def train_logistic_model(X, Y, iterations, lr):
    """Trains the logistic model and returns weights, threshold, and updated data."""
    predictions, weights = train_logistic_regression(X, Y, iterations, lr)
    threshold = find_best_threshold(pd.DataFrame({
        'Survived': Y,
        'PredictedChance': predictions
    }))
    return weights, threshold, predictions
    
def predict_test(X, weights, threshold, passenger_id):
    """Predicts survival on the test set and returns submission-ready DataFrame."""
    predictions = generate_predictions(X, weights)
    result = (predictions >= threshold).astype('int')
    return pd.DataFrame({'PassengerID': passenger_id, 'Survived': result})

def evaluate_predictions(predictions, actual, threshold):
    """Prints model accuracy on given data."""
    predictions = (predictions >= threshold).astype('int')
    accuracy = (predictions == actual).mean()
    return accuracy

def main():
    #load data
    train_file = load_and_prepare_data('/kaggle/input/titanic/train.csv')
    test_file = load_and_prepare_data('/kaggle/input/titanic/test.csv')

    #define features to use(these combination of features gives most accurate
    #test results)
    numerical_features = ['Age', 'FarePerPerson']
    categorical_features = ['Sex', 'Pclass', 'Title']
    int_features = ['FamilySize']
    features = numerical_features + categorical_features + int_features

    #training parameters
    iterations = 5000
    lr = 0.05

    Y = train_file['Survived']
    X = train_file[features]

    #build pipeline and apply it to the training data
    pipeline = build_pipeline(numerical_features, categorical_features, int_features)
    X_processed = pipeline.fit_transform(X)

    #split training data into validation sets
    X_train, X_val, Y_train, Y_val = train_test_split(X_processed, Y,
                                                     test_size = 0.2,
                                                     random_state = 1)

    #train logistic regression model
    weights, threshold, train_preds = train_logistic_model(X_train, Y_train, iterations, lr)
    
    #evaluate on validation set
    val_preds = generate_predictions(X_val, weights)
    val_accuracy = evaluate_predictions(val_preds, Y_val, threshold)
    print(f"Validation Accuracy: {val_accuracy:.4f}")

    #prepare test data and generate predictions
    passenger_id = test_file['PassengerId']
    test_processed = pipeline.transform(test_file)

    #save data to file
    submission = predict_test(test_processed, weights, threshold, passenger_id)
    submission.to_csv('submission.csv', index = False)

if __name__ == "__main__":
    main()

Validation Accuracy: 0.7821
